# PRE-ANÁLISIS

In [242]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [243]:
datos = pd.read_csv("diabetic_data.csv")

In [244]:
# display(datos.head()) # primeras 5
# display(datos.dtypes) # tipos
# datos.describe(include='all') # analiza media, std etc
# display(datos.isnull().any()) # valores null
display(datos.info())
# datos.describe()
# display(datos)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

None

In [245]:
datos["readmitted"] = datos["readmitted"].astype("category")
datos["readmitted"] = datos["readmitted"].map({
    "<30": 1,
    ">30": 0,
    "NO": 0
})

In [246]:
# Reemplazar los valores "?" por NaN (not a number)
datos = datos.replace("?", np.nan)
datos = datos.replace("None", np.nan)


In [247]:
# Quitar IDs (innecesarios para el análisis) y columnas con muchos valores faltantes
datos = datos.drop(columns=['encounter_id', 'patient_nbr'])
datos = datos.drop(columns=['weight'])
datos = datos.drop(columns=['max_glu_serum'])
datos = datos.drop(columns=['A1Cresult'])
datos = datos.drop(columns=['medical_specialty'])
datos = datos.drop(columns=['payer_code'])


Codificar variables categóricas

In [248]:
datos["race"].value_counts() # V. Categórica sin orden, usamos one-hot encoding

race
Caucasian          76099
AfricanAmerican    19210
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

In [249]:
datos = pd.get_dummies(datos, columns=['race'], drop_first=True)

In [250]:
datos["gender"].value_counts() # V. Categórica sin orden, usamos one-hot encoding

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

In [251]:
datos = pd.get_dummies(datos, columns=['gender'], drop_first=True)

In [252]:
datos["age"].value_counts() # V. Categórica con orden, mapearé con valor medio de cada rango

age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

In [253]:
datos["age"] = datos["age"].map({
    "[0-10)": 5,
    "[10-20)": 15,
    "[20-30)": 25,
    "[30-40)": 35,
    "[40-50)": 45,
    "[50-60)": 55,
    "[60-70)": 65,
    "[70-80)": 75,
    "[80-90)": 85,
    "[90-100)": 95
})

In [254]:
display(datos['diag_1'].value_counts().head()) # |
display(datos['diag_2'].value_counts().head()) # | Como tiene tantas categorías, se agruparán por sistemas.
display(datos['diag_3'].value_counts().head()) # |

diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
Name: count, dtype: int64

diag_2
276    6752
428    6662
250    6071
427    5036
401    3736
Name: count, dtype: int64

diag_3
250    11555
401     8289
276     5175
428     4577
427     3955
Name: count, dtype: int64

In [255]:
def seleccionarDiagnostico(codigo):
    cod = str(codigo).strip()
    try:
        codigoFloat = float(cod)
        if 390 <= codigoFloat <= 459 or codigoFloat == 785:
            return 'Circulatorio'
        elif 460 <= codigoFloat <= 519 or codigoFloat == 786:
            return 'Respiratorio'
        elif 520 <= codigoFloat <= 579 or codigoFloat == 787:
            return 'Digestivo'
        elif 250 <= codigoFloat < 251: # Diabetes es 250.xx
            return 'Diabetes'
        elif 800 <= codigoFloat <= 999:
            return 'Lesiones'
        elif 710 <= codigoFloat <= 739:
            return 'Musculoesquelético'
        elif 580 <= codigoFloat <= 629 or codigoFloat == 788:
            return 'Genitourinario'
        elif 140 <= codigoFloat <= 239:
            return 'Neoplasias'
        else:
            return 'Otros'
    except:
        return "Otros"

for col in ['diag_1', 'diag_2', 'diag_3']:
    datos[col] = datos[col].apply(seleccionarDiagnostico)
    datos = pd.get_dummies(datos, columns=[col], drop_first=True)


In [256]:
datos.info() # V. Categórica sin orden, usamos one-hot encoding

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 68 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   age                        101766 non-null  int64 
 1   admission_type_id          101766 non-null  int64 
 2   discharge_disposition_id   101766 non-null  int64 
 3   admission_source_id        101766 non-null  int64 
 4   time_in_hospital           101766 non-null  int64 
 5   num_lab_procedures         101766 non-null  int64 
 6   num_procedures             101766 non-null  int64 
 7   num_medications            101766 non-null  int64 
 8   number_outpatient          101766 non-null  int64 
 9   number_emergency           101766 non-null  int64 
 10  number_inpatient           101766 non-null  int64 
 11  number_diagnoses           101766 non-null  int64 
 12  metformin                  101766 non-null  object
 13  repaglinide                101766 non-null  